# Extract URLs from JSON "text" fields and fetch DOI links

Author: Corentin Briat
* Made to scrape papers from the Seminar on Biological Control Systems Slack

This notebook:
1. Recursively scans all `.json` files under `ROOT_DIR`.
2. Extracts URLs from each JSON object's `text` field into a pandas DataFrame.
3. Fetches each URL and tries to detect DOI(s) (doi.org links or DOI strings) and stores them in the DataFrame.


## 0) Set directory
Set `ROOT_DIR` to the directory containing your JSON files (Slack-export-like).

In [1]:
from pathlib import Path

# EDIT THIS:
ROOT_DIR = Path("./").expanduser().resolve()

ROOT_DIR, ROOT_DIR.exists()

(PosixPath('/Users/corentinbriat/Downloads/Seminar on Biological Control Systems Slack export Jun 28 2021 - Jan 12 2026/papers'),
 True)

## 1) Imports

If you don't have BeautifulSoup installed, run the install cell.

In [1]:
# If needed:
# !pip install requests beautifulsoup4 lxml pandas

import json
import re
import time
from typing import Dict, Iterable, List, Optional, Tuple

import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse


## 2) URL extraction from text

Supports Slack link markup:
- `<https://example.com|label>`
- `<https://example.com>`

and plain URLs.

In [3]:
SLACK_LINK_RE = re.compile(r"<(https?://[^>|]+)(?:\|[^>]+)?>")
PLAIN_URL_RE = re.compile(r"(https?://[^\s<>()\"']+)", re.IGNORECASE)
TRAILING_PUNCT = ".,;:!?)\u2019\u201d\"'"

def extract_urls_from_text(text: str) -> List[str]:
    if not text or not isinstance(text, str):
        return []
    urls: List[str] = []
    urls.extend(m.group(1) for m in SLACK_LINK_RE.finditer(text))
    urls.extend(m.group(1) for m in PLAIN_URL_RE.finditer(text))
    cleaned = [u.rstrip(TRAILING_PUNCT) for u in urls]
    # Dedupe while preserving order
    seen = set()
    out = []
    for u in cleaned:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out

# extract_urls_from_text("<https://doi.org/10.1038/s41587-021-00950-3|doi>")

## 3) Scan JSON files and build DataFrame of URL hits

Assumes each `.json` file is a list of dicts (messages). Extracts URLs from each dict's `text` field.

In [4]:
def iter_json_files(root: Path) -> Iterable[Path]:
    if root.is_file() and root.suffix.lower() == ".json":
        yield root
        return
    for p in root.rglob("*.json"):
        if p.is_file():
            yield p

def load_json(path: Path):
    try:
        txt = path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        txt = path.read_text(encoding="latin-1")
    return json.loads(txt)

rows = []
for jf in iter_json_files(ROOT_DIR):
    try:
        data = load_json(jf)
    except Exception:
        continue
    if not isinstance(data, list):
        continue
    for i, item in enumerate(data):
        if not isinstance(item, dict):
            continue
        text = item.get("text", "")
        urls = extract_urls_from_text(text)
        for u in urls:
            rows.append({
                "url": u,
                "file": str(jf),
                "message_index": i,
                "ts": str(item.get("ts", "")),
                "user": str(item.get("user", "")),
            })

df_urls = pd.DataFrame(rows)
df_urls.head(), len(df_urls), df_urls["url"].nunique()

(                                                 url  \
 0                   https://arxiv.org/abs/2409.00034   
 1  https://www.biorxiv.org/content/10.1101/2024.0...   
 2  https://www.biorxiv.org/content/10.1101/2024.0...   
 3  https://www.biorxiv.org/content/10.1101/2024.1...   
 4  https://www.biorxiv.org/content/10.1101/2022.0...   
 
                                                 file  message_index  \
 0  /Users/corentinbriat/Downloads/Seminar on Biol...              0   
 1  /Users/corentinbriat/Downloads/Seminar on Biol...              0   
 2  /Users/corentinbriat/Downloads/Seminar on Biol...              0   
 3  /Users/corentinbriat/Downloads/Seminar on Biol...              0   
 4  /Users/corentinbriat/Downloads/Seminar on Biol...              0   
 
                   ts         user  
 0  1725700424.044759  U029THP4J8Y  
 1  1712677662.913899  U02684Y5BGW  
 2  1712677662.913899  U02684Y5BGW  
 3  1732997414.424169  U02684Y5BGW  
 4  1643374186.531939  U02684Y5BGW  ,

In [5]:
DOI_RE = re.compile(r"\b10\.\d{4,9}/[-._;()/:A-Z0-9]+\b", re.IGNORECASE)

META_NAME_KEYS = {
    "citation_doi",
    "dc.identifier",
    "dc.identifier.doi",
    "prism.doi",
}
META_PROPERTY_KEYS = {
    "citation_doi",
    "prism:doi",
    "dc.identifier",
}

def normalize_doi(doi: str) -> str:
    doi = doi.strip()
    doi = doi.replace("doi:", "").replace("DOI:", "").strip()
    doi = doi.rstrip(".,;:!?)\"'")

    # Common preprint version suffix: ...v1, v2
    doi = re.sub(r"(10\.\d{4,9}/[^\s]+?)v\d+$", r"\1", doi, flags=re.IGNORECASE)
    return doi

def doi_to_link(doi: str) -> str:
    return f"https://doi.org/{doi}"

def dedupe_keep_order(seq: List[str]) -> List[str]:
    seen = set()
    out = []
    for x in seq:
        if x and x not in seen:
            seen.add(x)
            out.append(x)
    return out

def extract_dois_from_url(url: str) -> Tuple[List[str], List[str], str]:
    """Return (doi_links, dois, hint)."""
    if not url:
        return [], [], ""
    found = [normalize_doi(m.group(0)) for m in DOI_RE.finditer(url)]
    found = dedupe_keep_order([d for d in found if d])
    if found:
        return [doi_to_link(d) for d in found], found, "found_in_url"
    return [], [], ""


In [6]:
def make_session():
    s = requests.Session()
    s.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
            "(KHTML, like Gecko) Chrome/122.0 Safari/537.36"
        ),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Connection": "keep-alive",
    })
    return s

session = make_session()


In [7]:
import json as _json

def extract_dois_from_html(html: str) -> Tuple[List[str], List[str], str]:
    doi_links = []
    dois = []
    hints = []

    soup = BeautifulSoup(html, "lxml")

    # ---- A) Meta tags (publisher-agnostic) ----
    for meta in soup.find_all("meta"):
        name = (meta.get("name") or "").strip().lower()
        prop = (meta.get("property") or "").strip().lower()
        content = (meta.get("content") or "").strip()

        if (name in META_NAME_KEYS) or (prop in META_PROPERTY_KEYS):
            for m in DOI_RE.finditer(content):
                d = normalize_doi(m.group(0))
                if d:
                    dois.append(d)
                    doi_links.append(doi_to_link(d))
                    hints.append("found_in_meta")

    # ---- B) doi.org links ----
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "doi.org/" in href.lower():
            m = re.search(r"doi\.org/(.+)$", href, re.IGNORECASE)
            if m:
                d = normalize_doi(m.group(1))
                if d:
                    dois.append(d)
                    doi_links.append(doi_to_link(d))
                    hints.append("found_in_doi_link")

    # ---- C) JSON-LD (Nature often includes DOI here) ----
    for script in soup.find_all("script", attrs={"type": "application/ld+json"}):
        try:
            txt = script.string or script.get_text() or ""
            txt = txt.strip()
            if not txt:
                continue
            data = _json.loads(txt)

            # JSON-LD can be dict or list; normalize to list
            items = data if isinstance(data, list) else [data]

            for obj in items:
                if not isinstance(obj, dict):
                    continue

                # Common schema keys where DOI may appear
                # identifier can be dict or list; same for isPartOf, mainEntity, etc.
                candidates = []

                # direct fields
                for k in ["doi", "DOI", "identifier", "sameAs", "url"]:
                    if k in obj:
                        candidates.append(obj[k])

                # walk nested structures lightly
                for k in ["mainEntity", "isPartOf", "about"]:
                    if k in obj:
                        candidates.append(obj[k])

                # flatten candidates to strings
                def walk(x):
                    if isinstance(x, str):
                        yield x
                    elif isinstance(x, dict):
                        for v in x.values():
                            yield from walk(v)
                    elif isinstance(x, list):
                        for v in x:
                            yield from walk(v)

                for s in walk(candidates):
                    # capture doi.org or raw DOI
                    if "doi.org/" in s.lower():
                        m = re.search(r"doi\.org/(.+)$", s, re.IGNORECASE)
                        if m:
                            d = normalize_doi(m.group(1))
                            if d:
                                dois.append(d)
                                doi_links.append(doi_to_link(d))
                                hints.append("found_in_jsonld")
                    else:
                        for m in DOI_RE.finditer(s):
                            d = normalize_doi(m.group(0))
                            if d:
                                dois.append(d)
                                doi_links.append(doi_to_link(d))
                                hints.append("found_in_jsonld")
        except Exception:
            # ignore broken JSON-LD blocks
            continue

    # ---- D) Nature-specific / plain text fallback ----
    # If DOI appears as plain text (e.g., "https://doi.org/..." absent),
    # searching visible text helps.
    if not dois:
        text = soup.get_text(" ", strip=True)
        for m in DOI_RE.finditer(text):
            d = normalize_doi(m.group(0))
            if d:
                dois.append(d)
                doi_links.append(doi_to_link(d))
                hints.append("found_in_visible_text")
                break  # usually one DOI is enough

    # ---- E) Regex over raw HTML as final catch-all ----
    if not dois:
        for m in DOI_RE.finditer(html):
            d = normalize_doi(m.group(0))
            if d:
                dois.append(d)
                doi_links.append(doi_to_link(d))
                hints.append("found_by_regex")
                break

    doi_links = dedupe_keep_order(doi_links)
    dois = dedupe_keep_order(dois)
    hints = dedupe_keep_order(hints)

    return doi_links, dois, ";".join(hints)


In [8]:
def is_arxiv(url: str) -> bool:
    host = (urlparse(url).netloc or "").lower()
    return "arxiv.org" in host

def fetch_html_limited(resp: requests.Response, max_bytes: int = 200_000_000) -> str:
    content = b""
    for chunk in resp.iter_content(chunk_size=8192):
        if not chunk:
            break
        content += chunk
        if len(content) >= max_bytes:
            break
    try:
        return content.decode("utf-8", errors="ignore")
    except Exception:
        return content.decode("latin-1", errors="ignore")

def fetch_and_find_dois(url: str,
                        session: requests.Session,
                        timeout: float = 20,
                        max_bytes: int = 200_000_000,
                        sleep_s: float = 0.0) -> Dict[str, object]:

    out = {
        "url": url,
        "http_status": None,
        "final_url": "",
        "doi_links": "",
        "dois": "",
        "source_hint": "",
        "fetch_error": "",
    }

    doi_links, dois, hint = extract_dois_from_url(url)
    hints = [hint] if hint else []

    try:
        r = session.get(url, allow_redirects=True, timeout=timeout, stream=True)
        out["http_status"] = int(r.status_code)
        out["final_url"] = str(r.url)

        # DOI from final URL too
        dl2, d2, hint2 = extract_dois_from_url(out["final_url"])
        doi_links += dl2
        dois += d2
        if hint2:
            hints.append(hint2)

        if out["http_status"] == 403:
            out["fetch_error"] = "403 Forbidden (site may block automated requests)"
        else:
            html = fetch_html_limited(r, max_bytes=max_bytes)
            dl3, d3, hint3 = extract_dois_from_html(html)
            doi_links += dl3
            dois += d3
            if hint3:
                hints.append(hint3)

        r.close()

    except Exception as e:
        out["fetch_error"] = str(e)

    doi_links = dedupe_keep_order(doi_links)
    dois = dedupe_keep_order(dois)

    # If arXiv and no DOI, label it (arXiv often has none)
    if is_arxiv(url) and not dois:
        hints.append("arxiv_no_doi_expected")

    out["doi_links"] = ",".join(doi_links)
    out["dois"] = ",".join(dois)
    out["source_hint"] = ",".join(dedupe_keep_order([h for h in hints if h]))

    if sleep_s:
        time.sleep(sleep_s)

    return out


In [9]:
session = make_session()

nature_url = "https://www.nature.com/articles/s41467-024-46755-1"
res = fetch_and_find_dois(nature_url, session=session, sleep_s=0.0)
res


{'url': 'https://www.nature.com/articles/s41467-024-46755-1',
 'http_status': 200,
 'final_url': 'https://www.nature.com/articles/s41467-024-46755-1',
 'doi_links': 'https://doi.org/10.1038/s41467-024-46755-1,https://doi.org/10.15252%2Fmsb.20145735,https://doi.org/10.1126%2Fscience.aac7341,https://doi.org/10.1038%2Fncomms15459,https://doi.org/10.1038%2Fs41467-019-12706-4,https://doi.org/10.1038%2Fs41467-020-18302-1,https://doi.org/10.1038%2Fs41587-020-0468-5,https://doi.org/10.1126%2Fscience.1067407,https://doi.org/10.1016%2Fj.cell.2011.01.030,https://doi.org/10.1038%2Fs41589-018-0168-3,https://doi.org/10.1038%2Fnmeth.4505,https://doi.org/10.3389%2Ffbioe.2019.00080,https://doi.org/10.1038%2Fs41467-020-17993-w,https://doi.org/10.1021%2Facssynbio.5b00286,https://doi.org/10.1038%2Fmsb.2008.24,https://doi.org/10.1038%2Fs41467-018-05046-2,https://doi.org/10.15252%2Fmsb.20209618,https://doi.org/10.1073%2Fpnas.1202344109,https://doi.org/10.1126%2Fscience.1232758,https://doi.org/10.1038%2Fnbt.

In [10]:
# session = make_session()

nature_url = "https://www.cell.com/cell-systems/fulltext/S2405-4712%2816%2930040-0"
res = fetch_and_find_dois(nature_url, session=session, sleep_s=0.0)
res


{'url': 'https://www.cell.com/cell-systems/fulltext/S2405-4712%2816%2930040-0',
 'http_status': 403,
 'final_url': 'https://www.cell.com/cell-systems/fulltext/S2405-4712%2816%2930040-0',
 'doi_links': '',
 'dois': '',
 'source_hint': '',
 'fetch_error': '403 Forbidden (site may block automated requests)'}

In [12]:
FETCH_UNIQUE_ONLY = True
MAX_URLS = None   # set e.g. 100 for testing
SLEEP_S = 0.0     # set e.g. 0.2 if you want to be polite

if df_urls.empty:
    doi_df = pd.DataFrame(columns=["url","http_status","final_url","doi_links","dois","source_hint","fetch_error"])
else:
    urls = df_urls["url"].dropna().tolist()
    if FETCH_UNIQUE_ONLY:
        urls = pd.Series(urls).drop_duplicates().tolist()
    if MAX_URLS is not None:
        urls = urls[:MAX_URLS]

    results = [fetch_and_find_dois(u, session=session, sleep_s=SLEEP_S) for u in urls]
    doi_df = pd.DataFrame(results)

df_out = df_urls.merge(doi_df, on="url", how="left")
df_out.head()

# df_out.to_csv('./df_out.csv', index=False)

,url,file,message_index,ts,user,http_status,final_url,doi_links,dois,source_hint,fetch_error
0,https://arxiv.org/abs/2409.00034,/Users/corentinbriat/Downloads/Seminar on Biol...,0,1725700424.044759,U029THP4J8Y,200.0,https://arxiv.org/abs/2409.00034,"https://doi.org/10.1021/acssynbio.5c00099,http...","10.1021/acssynbio.5c00099,10.48550/arXiv.2409....",found_in_meta;found_in_doi_link,
1,https://www.biorxiv.org/content/10.1101/2024.0...,/Users/corentinbriat/Downloads/Seminar on Biol...,0,1712677662.913899,U02684Y5BGW,NaN,,https://doi.org/10.1101/2024.04.08.588465,10.1101/2024.04.08.588465,found_in_url,"HTTPSConnectionPool(host='www.biorxiv.org', po..."
2,https://www.biorxiv.org/content/10.1101/2024.0...,/Users/corentinbriat/Downloads/Seminar on Biol...,0,1712677662.913899,U02684Y5BGW,NaN,,https://doi.org/10.1101/2024.04.08.588465,10.1101/2024.04.08.588465,found_in_url,"HTTPSConnectionPool(host='www.biorxiv.org', po..."
3,https://www.biorxiv.org/content/10.1101/2024.1...,/Users/corentinbriat/Downloads/Seminar on Biol...,0,1732997414.424169,U02684Y5BGW,NaN,,https://doi.org/10.1101/2024.11.29.625997,10.1101/2024.11.29.625997,found_in_url,"HTTPSConnectionPool(host='www.biorxiv.org', po..."
4,https://www.biorxiv.org/content/10.1101/2022.0...,/Users/corentinbriat/Downloads/Seminar on Biol...,0,1643374186.531939,U02684Y5BGW,NaN,,https://doi.org/10.1101/2022.01.26.477951,10.1101/2022.01.26.477951,found_in_url,"HTTPSConnectionPool(host='www.biorxiv.org', po..."


In [13]:
df_out.to_csv('./df_out.csv', index=False)

In [1]:
import pandas as pd

df_out = pd.read_csv('./df_out.csv')
df_out.head(40)

,url,file,message_index,ts,user,http_status,final_url,doi_links,dois,source_hint,fetch_error
0,https://arxiv.org/abs/2409.00034,/Users/corentinbriat/Downloads/Seminar on Biol...,0,1.725700e+09,U029THP4J8Y,200.0,https://arxiv.org/abs/2409.00034,"https://doi.org/10.1021/acssynbio.5c00099,http...","10.1021/acssynbio.5c00099,10.48550/arXiv.2409....",found_in_meta;found_in_doi_link,NaN
1,https://www.biorxiv.org/content/10.1101/2024.0...,/Users/corentinbriat/Downloads/Seminar on Biol...,0,1.712678e+09,U02684Y5BGW,NaN,NaN,https://doi.org/10.1101/2024.04.08.588465,10.1101/2024.04.08.588465,found_in_url,"HTTPSConnectionPool(host='www.biorxiv.org', po..."
2,https://www.biorxiv.org/content/10.1101/2024.0...,/Users/corentinbriat/Downloads/Seminar on Biol...,0,1.712678e+09,U02684Y5BGW,NaN,NaN,https://doi.org/10.1101/2024.04.08.588465,10.1101/2024.04.08.588465,found_in_url,"HTTPSConnectionPool(host='www.biorxiv.org', po..."
3,https://www.biorxiv.org/content/10.1101/2024.1...,/Users/corentinbriat/Downloads/Seminar on Biol...,0,1.732997e+09,U02684Y5BGW,NaN,NaN,https://doi.org/10.1101/2024.11.29.625997,10.1101/2024.11.29.625997,found_in_url,"HTTPSConnectionPool(host='www.biorxiv.org', po..."
4,https://www.biorxiv.org/content/10.1101/2022.0...,/Users/corentinbriat/Downloads/Seminar on Biol...,0,1.643374e+09,U02684Y5BGW,NaN,NaN,https://doi.org/10.1101/2022.01.26.477951,10.1101/2022.01.26.477951,found_in_url,"HTTPSConnectionPool(host='www.biorxiv.org', po..."
5,https://www.biorxiv.org/content/10.1101/2022.0...,/Users/corentinbriat/Downloads/Seminar on Biol...,0,1.643374e+09,U02684Y5BGW,NaN,NaN,https://doi.org/10.1101/2022.01.26.477951,10.1101/2022.01.26.477951,found_in_url,"HTTPSConnectionPool(host='www.biorxiv.org', po..."
6,https://www.biorxiv.org/content/10.1101/2022.1...,/Users/corentinbriat/Downloads/Seminar on Biol...,0,1.669627e+09,U02684Y5BGW,NaN,NaN,https://doi.org/10.1101/2022.11.27.518074,10.1101/2022.11.27.518074,found_in_url,"HTTPSConnectionPool(host='www.biorxiv.org', po..."
7,https://www.biorxiv.org/content/10.1101/2022.1...,/Users/corentinbriat/Downloads/Seminar on Biol...,0,1.669627e+09,U02684Y5BGW,NaN,NaN,https://doi.org/10.1101/2022.11.27.518074,10.1101/2022.11.27.518074,found_in_url,"HTTPSConnectionPool(host='www.biorxiv.org', po..."
8,https://www.biorxiv.org/content/10.1101/2022.1...,/Users/corentinbriat/Downloads/Seminar on Biol...,1,1.669669e+09,U02684Y5BGW,NaN,NaN,https://doi.org/10.1101/2022.11.28.518161,10.1101/2022.11.28.518161,found_in_url,"HTTPSConnectionPool(host='www.biorxiv.org', po..."
9,https://www.biorxiv.org/content/10.1101/2022.1...,/Users/corentinbriat/Downloads/Seminar on Biol...,1,1.669669e+09,U02684Y5BGW,NaN,NaN,https://doi.org/10.1101/2022.11.28.518161,10.1101/2022.11.28.518161,found_in_url,"HTTPSConnectionPool(host='www.biorxiv.org', po..."


In [ ]:
# # We get all dois
# import numpy as np


# all_dois = []
# all_doistxt = ''

# for idx, row in df_out.iterrows():
#     print(idx,row['doi_links'])
#     temp_ = row['doi_links']
    
#     if str(temp_) != 'nan':
#         all_dois.append(row['doi_links'].replace(';', ', '))
#         if idx==0:
#             all_doistxt = row['doi_links'].replace(';', ', ')
#         else:
#             all_doistxt = all_doistxt + ', ' + row['doi_links'].replace(';', ', ')
#     # dois = row['dois'].replace(';', ',')

# all_doistxt
# with open("all.txt", "w") as text_file:
#     text_file.write(all_doistxt)

0 https://doi.org/10.1021/acssynbio.5c00099,https://doi.org/10.48550/arXiv.2409.00034
1 https://doi.org/10.1101/2024.04.08.588465
2 https://doi.org/10.1101/2024.04.08.588465
3 https://doi.org/10.1101/2024.11.29.625997
4 https://doi.org/10.1101/2022.01.26.477951
5 https://doi.org/10.1101/2022.01.26.477951
6 https://doi.org/10.1101/2022.11.27.518074
7 https://doi.org/10.1101/2022.11.27.518074
8 https://doi.org/10.1101/2022.11.28.518161
9 https://doi.org/10.1101/2022.11.28.518161
10 https://doi.org/10.1098/rsif.2023.0244,https://doi.org/10.48550/arXiv.2302.12521
11 nan
12 https://doi.org/10.1101/2024.03.25.586624
13 https://doi.org/10.1101/2024.03.25.586624
14 https://doi.org/10.1038/s41586-022-04470-1,https://doi.org/10.1038/s41586-022-04655-8,https://doi.org/10.2210/pdb7S4U/pdb,https://doi.org/10.2210/pdb7S4V/pdb,https://doi.org/10.2210/pdb7S4X/pdb,https://doi.org/10.7554%2FeLife.00471,https://doi.org/10.1126%2Fscience.1231143,https://doi.org/10.1038%2Fnbt.2623,https://doi.org/10.1038%2

In [16]:
# We get first dois

first_dois = []
first_doistxt = ''

for idx, row in df_out.iterrows():
    # print(row['doi_links'], '|', row['doi_links'].split(',')[0], '\n')
    # print(row['url'].split('|')[0])

    # print(idx,row['url'])
    temp_ = row['url']
    
    if str(temp_) != 'nan':

        url_ = row['url'].split('|')[0]
        # url_ = row['final_url'].split('%')[0].split('?')[0]

        if 'arxiv.org' in url_:
            first_dois.append(url_)
            first_doi = url_
        else:
            # print(idx, row['doi_links'])
            temp_ = row['doi_links']

            if str(temp_) != 'nan':
                first_doi = row['doi_links'].split(',')[0]
                first_dois.append(first_doi)
        
        if len(first_doi)>0:
            if len(first_doistxt)==0:
                first_doistxt = first_doi
            else:
                first_doistxt = first_doistxt + ', ' + first_doi
        # dois = row['dois'].replace(';', ',')

first_doistxt
with open("first.txt", "w") as text_file:
    text_file.write(first_doistxt)

In [ ]:
arxiv_urls = []
arxiv_urlstxt = ''
for idx, row in df_out.iterrows():
    url_ = row['final_url'].split('%')[0].split('?')[0]
    if 'arxiv.org' in url_:
        # print(url_)
        if len(arxiv_urlstxt)==0:
            arxiv_urlstxt = url_
        else:
            arxiv_urlstxt = arxiv_urlstxt + ', ' + url_

    # for url_ in url_tab:
    #     if 'arxiv.org' in url_:
    #         if len(all_urlstxt)==0:
    #             all_urlstxt = url_
    #         else:
    #             all_urlstxt = url_ + ', ' + url_

# all_doistxt
with open("arxiv.txt", "w") as text_file:
    text_file.write(arxiv_urlstxt)

In [ ]:
if not df_out.empty:
    has_doi = df_out["dois"].fillna("").str.len() > 0
    print("Rows with DOI:", int(has_doi.sum()), "/", len(df_out))
    print("Unique URLs with DOI:", df_out.loc[has_doi, "url"].nunique(), "/", df_out["url"].nunique())
    df_out.loc[~has_doi, ["url","http_status","fetch_error","source_hint"]].head(20)
